# Retail Banking Customer Churn & Retention Analytics\n**Emmanuel Odinamba** | Python · SQL · Statistical Analysis · Machine Learning\n\nThis notebook analyzes a public 10,000-customer banking dataset to identify churn patterns and build an interpretable churn-risk model.

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport sqlite3\n\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay\n

## 1. Load and inspect the data

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/YBI-Foundation/Dataset/main/Bank%20Churn%20Modelling.csv'\ndf = pd.read_csv(DATA_URL)\nprint('Shape:', df.shape)\ndisplay(df.head())\n

## 2. Data-quality review\nBefore analysis, check missing values, duplicates and data types.

In [ ]:
print('Missing values:', df.isna().sum().sum())\nprint('Duplicate rows:', df.duplicated().sum())\ndisplay(df.dtypes.to_frame('dtype'))\n\ndf = df.rename(columns={'Num Of Products':'NumOfProducts','Has Credit Card':'HasCreditCard','Is Active Member':'IsActiveMember','Estimated Salary':'EstimatedSalary'})\ndf = df.drop(columns=['CustomerId','Surname'], errors='ignore')\n

## 3. Exploratory analysis\nMeasure the portfolio churn KPI and identify segments with elevated churn.

In [ ]:
print(f'Overall churn rate: {df.Churn.mean():.2%}')\nfor col in ['Geography','Gender','IsActiveMember','NumOfProducts']:\n    out=df.groupby(col)['Churn'].agg(customers='size',churn_rate='mean')\n    out['churn_rate']=(out['churn_rate']*100).round(2)\n    display(out.sort_values('churn_rate',ascending=False))\n

In [ ]:
df['AgeBand']=pd.cut(df.Age,[17,29,45,60,100],labels=['Under 30','30-45','46-60','60+'])\nage=df.groupby('AgeBand',observed=False)['Churn'].agg(customers='size',churn_rate='mean')\nage['churn_rate']=(age.churn_rate*100).round(2)\ndisplay(age)\n

## 4. SQL analysis\nLoad the analytical table into SQLite and answer the same business questions using SQL.

In [ ]:
conn=sqlite3.connect(':memory:')\ndf.drop(columns=['AgeBand']).to_sql('customers',conn,index=False,if_exists='replace')\nquery='''SELECT Geography, COUNT(*) AS customers, ROUND(100.0*AVG(Churn),2) AS churn_rate_pct, ROUND(AVG(Balance),2) AS avg_balance FROM customers GROUP BY Geography ORDER BY churn_rate_pct DESC;'''\ndisplay(pd.read_sql_query(query,conn))\n

## 5. Interpretable churn model\nUse a stratified 80/20 split and logistic regression. Class weighting helps account for the smaller churn class.

In [ ]:
X=df.drop(columns=['Churn','AgeBand'])\ny=df.Churn\ncat=['Geography','Gender']\nnum=[c for c in X.columns if c not in cat]\npre=ColumnTransformer([('num',StandardScaler(),num),('cat',OneHotEncoder(handle_unknown='ignore'),cat)])\nmodel=Pipeline([('preprocess',pre),('model',LogisticRegression(max_iter=1000,class_weight='balanced',random_state=42))])\nX_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)\nmodel.fit(X_train,y_train)\npred=model.predict(X_test)\nprob=model.predict_proba(X_test)[:,1]\nprint(classification_report(y_test,pred,digits=3))\nprint('ROC-AUC:',round(roc_auc_score(y_test,prob),3))\n

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test,pred)\nplt.title('Churn Model Confusion Matrix')\nplt.show()\nRocCurveDisplay.from_predictions(y_test,prob)\nplt.title('ROC Curve — Customer Churn Model')\nplt.show()\n

## 6. Interpretation and business use\nThe model is a prioritization tool, not proof of causality. Segment-level findings should trigger deeper investigation. A practical retention workflow would combine churn probability with customer value, then test targeted interventions and measure incremental retention.